# AI4I — F1/F2 fintzea eta Isolation Forest

Balioztatze eta atalase-hautaketa emaitzak; ez azken test independentea.
25 RF konfigurazio × 4 CV fold. Lehengo test errenkadak baztertuta.
Erreprodukzioa: `python -m scripts.refine_model --output models/offline-ai4i-v2`.

In [1]:
from pathlib import Path
import json, sys
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / 'data/ai4i2020_train.csv').exists(): ROOT = ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from scoring.inference import load_bundle, predict_signals
run = ROOT / 'models/offline-ai4i-v2'
report = json.loads((run / 'report.json').read_text())
bundle = load_bundle(run)
print(report['versions'])

{'python': '3.14.8', 'sklearn': '1.9.1', 'pandas': '3.0.0', 'numpy': '2.4.2', 'joblib': '1.6.0'}


## Hautaketa eta benetako akatsak

F2-k recall lehenesten du F1ek baino gehiago. Ez da berariazko kostu ekonomikoaren ordezkoa.
OOF balioek hiperparametro-hautaketa barne dute; balioztatzeak atalase-hautaketa. Ez dira nested CV edo kanpoko test emaitzak.

In [2]:
for policy, result in report['selected'].items():
    print(policy, json.dumps(result, indent=2))
    candidate = report['search'][result['candidate_id']]
    print('Parameters:', candidate['parameters'], 'engineered:', candidate['engineered'])

f1 {
  "candidate_id": 21,
  "validation_threshold_tuning": {
    "threshold": 0.5506458333333333,
    "precision": 1.0,
    "recall": 0.7317073170731707,
    "f1": 0.8450704225352113,
    "f2": 0.7731958762886598,
    "average_precision": 0.8669517684901401,
    "tn": 1159,
    "fp": 0,
    "fn": 11,
    "tp": 30
  }
}
Parameters: {'min_samples_leaf': 1, 'max_features': 1.0, 'max_depth': 8, 'class_weight': None} engineered: True
f2 {
  "candidate_id": 18,
  "validation_threshold_tuning": {
    "threshold": 0.3679497423165187,
    "precision": 0.9117647058823529,
    "recall": 0.7560975609756098,
    "f1": 0.8266666666666667,
    "f2": 0.7828282828282829,
    "average_precision": 0.8615312293465427,
    "tn": 1156,
    "fp": 3,
    "fn": 10,
    "tp": 31
  }
}
Parameters: {'min_samples_leaf': 5, 'max_features': 1.0, 'max_depth': 8, 'class_weight': None} engineered: True


## Isolation Forest: seinale bereizia

Normal gisa etiketatutako entrenamendu-erregistroak bakarrik. Atalasea balioztatze normalaren 99. pertzentila.
Hutsegitea ez da anomalia baten etiketa egiaztatua; ez dago IFk RF hobetzen duen frogarik.

In [3]:
print({k:v for k,v in report['isolation_forest'].items() if k != 'fit_row_indices'})

{'threshold': 0.6338224044766037, 'normal_validation_quantile': 0.99, 'normal_validation_rows': 1159, 'validation_diagnostic_against_failure_not_anomaly_truth': {'threshold': 0.6338224044766037, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'f2': 0.0, 'average_precision': 0.08186312599787121, 'tn': 1147, 'fp': 12, 'fn': 41, 'tp': 0}, 'normal_proxy': 'training machine_failure == 0', 'parameters': {'bootstrap': False, 'contamination': 'auto', 'max_features': 1.0, 'max_samples': 256, 'n_estimators': 200, 'n_jobs': 1, 'random_state': 42, 'verbose': 0, 'warm_start': False}, 'observed_normal_validation_alert_fraction': 0.010353753235547885}


## Inferentzia partekatua, sensor payload batekin

Arrisku- eta anomalia-abisuak independenteak dira. F1 lehenetsia, F2 aukeran. Ez dago makina gelditzeko agindurik.

In [4]:
sample = pd.DataFrame([{'machine_type':'L', 'air_temp_K':300.0, 'process_temp_K':310.0, 'rpm':1500.0, 'torque_Nm':40.0, 'tool_wear_min':100.0}])
for policy in ['f1','f2']:
    print(predict_signals(bundle, sample, policy))

[{'failure_score': 0.00027020644452422667, 'risk_alert': False, 'anomaly_score': 0.40460772273402684, 'anomaly_alert': False, 'policy': 'f1', 'model_version': 'offline-ai4i-v2'}]
[{'failure_score': 0.0001501274763479009, 'risk_alert': False, 'anomaly_score': 0.40460772273402684, 'anomaly_alert': False, 'policy': 'f2', 'model_version': 'offline-ai4i-v2'}]


## Ondorioak

F1 politika: 30/41 detektatuta, FP=0, FN=11. F2 politika: 31/41, FP=3, FN=10.
IFk 12 normal markatu ditu eta hutsegiteetako 0 (%1eko atalase esploratzailean).
Ez dago 24 orduko horizontea, kalibrazioa, alarma/egun edo industria-eraginkortasuna egiaztatzeko daturik.
Xehetasunak eta mugak: `docs/data/FIN_SINTONIA.md`.